# v14: Wu + Berke, CE + lambda x population beta loss (v8 regime), Kev 800M

**Comparison with v8** (`train-beta-v8-wu-berke-sample-lrscale-800m`). Same model, data, sample size, LR, epochs, seed,
validators and best.pt rule as v8; only the beta term changes: v8's sample beta loss is replaced by the population beta
loss. Benchmarks: v8's CE-only run (lambda = 0) and v8's sample-loss runs (lambda = 0.25 and 1).

In [1]:
import math
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

# Walk up from the notebook folder to locate the repository's `code/` root
CODE_DIR = next(
    (c for p in (Path.cwd(), *Path.cwd().parents) for c in (p, p / "code") if all((c / d).is_dir() for d in ("kev", "data", "models"))),
    None,
)
if CODE_DIR is None:
    raise RuntimeError(f"No code/ root containing kev/, data/, and models/ at or above {Path.cwd()}")

# The `beta_ols`, `beta_train` and `beta_ce_population` packages are in `src/` next to this notebook
EXPLORATION_DIR = CODE_DIR / "papers" / "beta-loss" / "exploration"
SRC_DIR = EXPLORATION_DIR / "src"
if not (SRC_DIR / "beta_ce_population").is_dir():
    raise RuntimeError(f"`beta_ce_population` package not found in {SRC_DIR}")

# Run from `code/` and expose `code/`, `code/kev/`, and `src/` on sys.path
os.chdir(CODE_DIR)
for _p in (str(CODE_DIR), str(CODE_DIR / "kev"), str(SRC_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from models.kev_pointer import KevPointerModel
from training import free_gpu

import beta_ols
from beta_ols import core as beta_core
from beta_ce_population.config import CEPopulationTrainConfig
from beta_ce_population.trainer import load_ce_population_run, run_ce_population_training
from beta_train.config import scale_lr_for_sample_size
from beta_train.diagnostics import find_max_micro_chunk
from beta_train.items import build_items_cached, items_summary
from beta_train.model_state import set_checkpointing, snapshot_trainable
from beta_train.trainer import load_run
from beta_train.validation import PopulationValidator, validate_all_cached, validation_metrics_frame

# Disk cache for training items and the untrained (step-0) validation; delete the folder to clear it
CACHE_DIR = CODE_DIR / ".cache" / "beta_training"

print(f"CODE_DIR: {CODE_DIR}")

CODE_DIR: /home/dw/github/decision-models-finetuning/code


In [ ]:
# Require CUDA and release any previously loaded `kev_model` before allocating a fresh 800M backbone
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required to train jaredpalmer/kev-0.8b in bf16")
if "kev_model" in globals():
    globals().pop("kev_model")
device = "cuda"
free_gpu()
torch.manual_seed(42)

# Model settings (also the key of the cached step-0 validation)
KEV_BASE = "jaredpalmer/kev-0.8b"
LORA_RANK = 64
WEIGHTS_DTYPE = "bf16"
HEAD_TEMPERATURE = 1.0   # training uses raw uncalibrated temperature = 1.0
# Recompute activations in backward (less VRAM, slower). Memory-only: not part of a run's identity
GRADIENT_CHECKPOINTING = False

# Build the frozen backbone + LoRA + pointer head
t_model0 = time.perf_counter()
kev_model = KevPointerModel.build(
    KEV_BASE,
    device=device,
    lora_rank=LORA_RANK,
    weights_dtype=WEIGHTS_DTYPE,
)
kev_model.net.head.temperature = HEAD_TEMPERATURE
kev_model.train()                     # enable training mode (LoRA dropout active, trainable params require_grad)
set_checkpointing(kev_model, GRADIENT_CHECKPOINTING)
torch.cuda.synchronize()
model_load_sec = time.perf_counter() - t_model0

trainable_params_count = sum(p.numel() for p in kev_model.trainable_parameters())
total_params_count = sum(p.numel() for p in kev_model.parameters())
display(
    pd.Series(
        {
            "base_model": KEV_BASE,
            "mode": kev_model.mode,
            "training": kev_model.training,
            "weights_dtype": kev_model.meta.weights_dtype,
            "lora_rank": kev_model.meta.lora,
            "trainable_params": f"{trainable_params_count:,}",
            "total_params": f"{total_params_count:,}",
            "gradient_checkpointing": GRADIENT_CHECKPOINTING,
            "model_load_sec": round(model_load_sec, 3),
        },
        name="kev_800m_train_mode",
    ).to_frame()
)

# Snapshot the untrained trainable weights right after building, so every run and the step-0 validation start from them
INIT_TRAINABLE_STATE = snapshot_trainable(kev_model)
MODEL_KEY = {"base": KEV_BASE, "lora_rank": LORA_RANK, "weights_dtype": WEIGHTS_DTYPE, "head_temperature": HEAD_TEMPERATURE}
print(f"Snapshotted {len(INIT_TRAINABLE_STATE)} trainable tensors")

# Datasets (loaded and replicated through `beta_ols`)

In [ ]:
# v8's design: train on Wu and Berke (CE on their y, plus their population betas), validate on the held-out Kriner
# and Grummon 2025
ITEM_DATASETS = [         # training items are built for these
    "wu2025",
    "berke2023",
]
VALIDATION_DATASETS = [   # validation runs on these
    "kriner2026",
    "grummon2025",
]
LOADED_DATASETS = list(ITEM_DATASETS)
for dataset_name in VALIDATION_DATASETS:
    if dataset_name not in LOADED_DATASETS:
        LOADED_DATASETS.append(dataset_name)

# Load each through its `beta_ols` module: task rows, regression rows, regression specs, human OLS replication.
# Training uses one piece per task variable: Wu is one piece (its own name), Berke four (one per delivery task).
choice_dfs: dict[str, pd.DataFrame] = {}
regression_rows_by_dataset: dict[str, pd.DataFrame] = {}
regression_specs_by_dataset: dict[str, dict] = {}
human_ols_by_dataset: dict[str, pd.DataFrame] = {}
module_by_dataset = {}    # (pseudo) dataset name -> the beta_ols module that codes its rows
TRAIN_DATASETS = []       # pseudo datasets of ITEM_DATASETS, the names training uses

for dataset_name in LOADED_DATASETS:
    dataset_module = beta_ols.DATASET_MODULES[dataset_name]
    full_choice_df = dataset_module.load_choice_df()
    pieces = {}
    if dataset_name in VALIDATION_DATASETS:
        pieces[dataset_name] = full_choice_df
    if dataset_name in ITEM_DATASETS:
        task_pieces = beta_core.split_by_task(full_choice_df, dataset_name)
        pieces.update(task_pieces)
        TRAIN_DATASETS.extend(task_pieces)

    for piece_name, choice_df in pieces.items():
        rows = dataset_module.regression_rows(choice_df)
        specs = dataset_module.regression_specs(rows)
        module_by_dataset[piece_name] = dataset_module
        choice_dfs[piece_name] = choice_df
        regression_rows_by_dataset[piece_name] = rows
        regression_specs_by_dataset[piece_name] = specs
        human_ols_by_dataset[piece_name] = beta_core.fit_human_ols(rows, specs)
        print(f"{piece_name}: {choice_df['CASEID'].nunique():,} respondents | {len(choice_df):,} tasks | {len(specs)} regression(s)")
        display(beta_core.wide_ols_table(human_ols_by_dataset[piece_name]).round(4))
print(f"TRAIN_DATASETS = {TRAIN_DATASETS}")

# Population beta targets (standardized human population betas of each training piece)

In [ ]:
# One target vector per training piece and regression: the standardized human population betas (beta / sigma),
# in the spec's target order, from the human OLS replication of that piece (a published-table stand-in)
target_std_betas_by_dataset = {}
for dataset_name in TRAIN_DATASETS:
    dataset_ols = human_ols_by_dataset[dataset_name]
    dataset_targets = {}
    for group, spec in regression_specs_by_dataset[dataset_name].items():
        group_ols = dataset_ols[dataset_ols["model"] == group].set_index("term")
        target_values = group_ols.loc[spec.target_labels, "std_beta"].to_numpy(dtype=np.float64)
        dataset_targets[group] = torch.tensor(target_values, dtype=torch.float64)
    target_std_betas_by_dataset[dataset_name] = dataset_targets

for dataset_name, dataset_targets in target_std_betas_by_dataset.items():
    for group, target in dataset_targets.items():
        labels = regression_specs_by_dataset[dataset_name][group].target_labels
        display(pd.Series(target.numpy(), index=labels, name=f"{dataset_name} | {group} target std beta").round(4).to_frame())

# Validators and training items

In [ ]:
# One fused validator per VALIDATION_DATASETS entry (population OLS from the dataset's module)
# Per-GPU-batch inference budgets: lower them if validation runs out of memory, raise for speed
VALIDATION_MAX_QUESTIONS = 1024
VALIDATION_MAX_CHARS = 1_000_000

validators = {}
for dataset_name in VALIDATION_DATASETS:
    validators[dataset_name] = PopulationValidator(
        beta_ols.DATASET_MODULES[dataset_name],
        choice_dfs[dataset_name],
        regression_specs_by_dataset[dataset_name],
        max_questions=VALIDATION_MAX_QUESTIONS,
        max_chars=VALIDATION_MAX_CHARS,
    )
display(pd.DataFrame([validator.describe() for validator in validators.values()]).set_index("validator"))

In [ ]:
# Untrained (step-0) validation, loaded from the disk cache when it already ran for this model, data, and code;
# reused as the first validation row of every run
step0_validation, step0_cache_report = validate_all_cached(
    kev_model,
    validators,
    INIT_TRAINABLE_STATE,
    model_key=MODEL_KEY,
    cache_dir=CACHE_DIR,
)
display(step0_cache_report)
display(validation_metrics_frame(step0_validation).round(4))
for dataset_name, (_, beta_df) in step0_validation.items():
    display(beta_df.round(4))

In [ ]:
# Tokenize the TRAIN_DATASETS rows into training items (one per data row), loaded from the disk cache when they were
# already built for this data, tokenizer, and code
item_modules = {}
for dataset_name in TRAIN_DATASETS:
    item_modules[dataset_name] = module_by_dataset[dataset_name]
unified_items, items_cache_report = build_items_cached(kev_model, item_modules, choice_dfs, cache_dir=CACHE_DIR)
display(items_cache_report)
display(items_summary(unified_items, regression_specs_by_dataset))

# Find max micro chunk

In [ ]:
# Largest micro-chunk (items = data rows per chunk) that fits in VRAM for the worst-case (most padded) training chunks
SEARCH_GRADIENT_CHECKPOINTING = GRADIENT_CHECKPOINTING   # search with the setting you train with
SEARCH_MAX_CHUNK = 64
SEARCH_CANDIDATES_PER_SIZE = 3          # worst-case chunks tested per size; a size fits only if all of them fit

search_items = []
for dataset_name in TRAIN_DATASETS:
    search_items.extend(unified_items[dataset_name])
micro_chunk_search_df, MAX_FITTING_MICRO_CHUNK = find_max_micro_chunk(
    kev_model,
    search_items,
    INIT_TRAINABLE_STATE,
    max_chunk=SEARCH_MAX_CHUNK,
    gradient_checkpointing=SEARCH_GRADIENT_CHECKPOINTING,
    candidates_per_size=SEARCH_CANDIDATES_PER_SIZE,
)
display(micro_chunk_search_df)
print(f"gradient_checkpointing={SEARCH_GRADIENT_CHECKPOINTING} | MAX_FITTING_MICRO_CHUNK = {MAX_FITTING_MICRO_CHUNK}")

# Train

In [ ]:
# Run folders: one per run under RUNS_DIR (stats, betas per validation, checkpoints/best.pt, checkpoints/last.pt)
NOTEBOOK = "train-beta-v14-wu-berke-ce-population-800m"
RUNS_DIR = EXPLORATION_DIR / "runs" / NOTEBOOK
# ONE validator's column picks checkpoints/best.pt (v8's rule)
BEST_METRIC = "kriner2026/val_loss"
BEST_MODE = "min"

from_scratch = False

# v8's regime: S = 40 tasks per sample, peak LR = BASE_LR_PER_TASK x sqrt(S) = 4e-5, one epoch, OneCycle
SAMPLE_SIZE = 40
NUM_EPOCHS = 1
BASE_LR_PER_TASK = 4e-5 / math.sqrt(40)
PEAK_LR = scale_lr_for_sample_size(BASE_LR_PER_TASK, SAMPLE_SIZE)
# Rows (training items) seen between validations, per validator (all validators also run at step 0 and the last step)
EVAL_EVERY_ROWS_BY_VALIDATOR = {
    "kriner2026": 1280,
    "grummon2025": 6400,
}
EVAL_EVERY_BY_VALIDATOR = {}
for validator_name, eval_every_rows in EVAL_EVERY_ROWS_BY_VALIDATOR.items():
    EVAL_EVERY_BY_VALIDATOR[validator_name] = max(1, round(eval_every_rows / SAMPLE_SIZE))
MICRO_CHUNK = MAX_FITTING_MICRO_CHUNK
# Items per no-grad eval-mode forward when a run computes its frozen base probabilities p0 (VRAM only; lower it on OOM)
P0_CHUNK = 32

# Settings recorded in each run's run.json (not used by training)
RUN_METADATA = {
    "notebook": NOTEBOOK,
    "model": MODEL_KEY,
    "item_datasets": ITEM_DATASETS,
    "validation_datasets": VALIDATION_DATASETS,
    "validator_kind": "fused",
    "eval_every_rows_by_validator": EVAL_EVERY_ROWS_BY_VALIDATOR,
    "base_lr_per_task": BASE_LR_PER_TASK,
}
runs = {}


In [ ]:
def train_ce_population(beta_lambda: float):
    # One CE + lambda x population beta run (total loss = CE + beta_lambda x population beta cross loss), every other
    # setting from the cell above; an existing run is resumed, or loaded if finished; from_scratch=True deletes it and
    # trains again. The result is stored in `runs` under the run folder name.
    run_config = CEPopulationTrainConfig(
        train_datasets=tuple(TRAIN_DATASETS),
        sample_size_by_dataset={dataset_name: SAMPLE_SIZE for dataset_name in TRAIN_DATASETS},
        beta_lambda=beta_lambda,
        lr=PEAK_LR,
        num_epochs=NUM_EPOCHS,
        eval_every_by_validator=EVAL_EVERY_BY_VALIDATOR,
        lr_schedule="onecycle",
        weight_decay=0.01,
        micro_chunk=MICRO_CHUNK,
        p0_chunk=P0_CHUNK,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        grad_clip=1.0,
        seed=42,
    )
    display(pd.Series(run_config.as_dict(), name=run_config.label()).to_frame())
    run_name = f"lambda={beta_lambda:g}_lr={PEAK_LR:g}_S={SAMPLE_SIZE}_ep={NUM_EPOCHS}"
    runs[run_name] = run_ce_population_training(
        model=kev_model,
        items_by_dataset=unified_items,
        specs_by_dataset=regression_specs_by_dataset,
        target_std_betas_by_dataset=target_std_betas_by_dataset,
        validators=validators,
        step0_validation=step0_validation,
        init_snapshot=INIT_TRAINABLE_STATE,
        config=run_config,
        device=device,
        run_dir=RUNS_DIR / run_name,
        best_metric=BEST_METRIC,
        best_mode=BEST_MODE,
        run_metadata=RUN_METADATA,
        from_scratch=from_scratch,
    )


## λ = 0.25: population beta loss vs v8's sample beta loss at the same λ

v8 benchmark (same regime; Kriner picks best.pt, Grummon is the held-out check):

| v8 run (best.pt by `kriner2026/val_loss`) | best step / 1,375 | Kriner KL | Kriner std beta MSE | Kriner ρ | Grummon KL | Grummon std beta MSE | Grummon ρ |
|---|---|---|---|---|---|---|---|
| untrained (step 0) | 0 | | 0.0710 | | | 0.0283 | |
| λ = 0 (CE only) | 192 | 0.6325 | 0.0160 | 0.783 | 0.8876 | 0.0206 | 0.6 |
| λ = 0.25 (CE + 0.25 × sample beta loss) | 288 | 0.6331 | 0.0175 | 0.769 | 0.9006 | 0.0202 | 0.8 |
| λ = 1 (CE + 1 × sample beta loss) | 928 | 0.6324 | **0.0135** | 0.734 | 0.8733 | 0.0162 | 0.6 |

To beat: Kriner std beta MSE below 0.0175 (v8 λ = 0.25) and 0.0160 (v8 λ = 0), at Kriner KL no worse than about 0.633.
Grummon has only 6 validations per run, so its numbers are noisy (ρ is over 5 terms).

In [ ]:
train_ce_population(0.25)


## λ = 1.0: population beta loss vs v8's sample beta loss at the same λ

v8 benchmark (same regime; Kriner picks best.pt, Grummon is the held-out check):

| v8 run (best.pt by `kriner2026/val_loss`) | best step / 1,375 | Kriner KL | Kriner std beta MSE | Kriner ρ | Grummon KL | Grummon std beta MSE | Grummon ρ |
|---|---|---|---|---|---|---|---|
| untrained (step 0) | 0 | | 0.0710 | | | 0.0283 | |
| λ = 0 (CE only) | 192 | 0.6325 | 0.0160 | 0.783 | 0.8876 | 0.0206 | 0.6 |
| λ = 0.25 (CE + 0.25 × sample beta loss) | 288 | 0.6331 | 0.0175 | 0.769 | 0.9006 | 0.0202 | 0.8 |
| λ = 1 (CE + 1 × sample beta loss) | 928 | 0.6324 | **0.0135** | 0.734 | 0.8733 | 0.0162 | 0.6 |

To beat: Kriner std beta MSE below 0.0135 (v8 λ = 1, the best v8 run on Kriner) and 0.0160 (v8 λ = 0), at Kriner KL no
worse than about 0.633. Grummon has only 6 validations per run, so its numbers are noisy (ρ is over 5 terms).

In [ ]:
train_ce_population(1.0)


# Compare with v8 (same regime, sample beta loss)

In [3]:
# Comparison read from the run folders on disk (needs only the first code cell, so it works without training in this
# session): one row per run, the validation of its best.pt. A run folder that does not exist yet is skipped with a
# warning; an unfinished run is shown with its best validation so far (status column).
import warnings

from beta_train.runs import read_run_json

OLD_LABEL = "v8"
OLD_RUNS_DIR = EXPLORATION_DIR / "runs" / "train-beta-v8-wu-berke-sample-lrscale-800m"
OLD_RUN_NAMES = [
    "lambda=0_lr=4e-05_S=40_ep=1",
    "lambda=0.25_lr=4e-05_S=40_ep=1",
    "lambda=1_lr=4e-05_S=40_ep=1",
]
NEW_RUNS_DIR = EXPLORATION_DIR / "runs" / "train-beta-v14-wu-berke-ce-population-800m"
NEW_RUN_NAMES = [
    "lambda=0.25_lr=4e-05_S=40_ep=1",
    "lambda=1_lr=4e-05_S=40_ep=1",
]
COMPARE_VALIDATORS = ["kriner2026", "grummon2025"]
COMPARE_METRICS = ["kl_loss", "std_beta_mse", "beta_mse", "spearman_rho"]

run_sources = []   # (row label, run folder, loader)
for run_name in OLD_RUN_NAMES:
    run_sources.append((f"{OLD_LABEL} sample beta | {run_name}", OLD_RUNS_DIR / run_name, load_run))
for run_name in NEW_RUN_NAMES:
    run_sources.append((f"v14 population beta | {run_name}", NEW_RUNS_DIR / run_name, load_ce_population_run))

comparison_rows = []
for label, run_dir, load_saved_run in run_sources:
    if not (run_dir / "run.json").exists():
        warnings.warn(f"No saved run in {run_dir}; it is left out of the comparison")
        continue
    run_result = load_saved_run(run_dir)
    row = {"run": label, "status": read_run_json(run_dir)["status"], "best_step": run_result.best["step"]}
    for validator_name in COMPARE_VALIDATORS:
        for metric in COMPARE_METRICS:
            row[f"{validator_name}/{metric}"] = run_result.best_row[f"{validator_name}/{metric}"]
    comparison_rows.append(row)
if comparison_rows:
    display(pd.DataFrame(comparison_rows).set_index("run").round(5))
else:
    warnings.warn("No saved runs to compare")


/tmp/ipykernel_170775/4030911147.py:32: UserWarning: No saved run in /home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/runs/train-beta-v14-wu-berke-ce-population-800m/lambda=0.25_lr=4e-05_S=40_ep=1; it is left out of the comparison
  warnings.warn(f"No saved run in {run_dir}; it is left out of the comparison")
/tmp/ipykernel_170775/4030911147.py:32: UserWarning: No saved run in /home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/runs/train-beta-v14-wu-berke-ce-population-800m/lambda=1_lr=4e-05_S=40_ep=1; it is left out of the comparison
  warnings.warn(f"No saved run in {run_dir}; it is left out of the comparison")


,status,best_step,kriner2026/kl_loss,kriner2026/std_beta_mse,kriner2026/beta_mse,kriner2026/spearman_rho,grummon2025/kl_loss,grummon2025/std_beta_mse,grummon2025/beta_mse,grummon2025/spearman_rho
run,,,,,,,,,,
v8 sample beta | lambda=0_lr=4e-05_S=40_ep=1,finished,192,0.63249,0.01601,0.00400,0.78322,0.88763,0.02061,0.00398,0.6
v8 sample beta | lambda=0.25_lr=4e-05_S=40_ep=1,finished,288,0.63309,0.01750,0.00438,0.76923,0.90063,0.02020,0.00390,0.8
v8 sample beta | lambda=1_lr=4e-05_S=40_ep=1,finished,928,0.63242,0.01352,0.00338,0.73427,0.87328,0.01622,0.00313,0.6
